In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
from pathlib import Path
import random
from tqdm import tqdm
import math


# Initialization

TARGET_BANDS = [6, 7, 10, 12, 16, 21]  # Change Band numbers if required
NUM_TARGET_BANDS = len(TARGET_BANDS)

RGB_BANDS = [20, 11, 5] # Change Band numbers if required
NUM_INPUT_BANDS = len(RGB_BANDS)

class Config:
    # Dataset paths
    train_data = 'Enter The train path/Hyperion/train_images'
    test_data = 'Enter The test path/Hyperion/test_images'
    water_bodies_dir = 'Evaluation/Hyperion/water_bodies'


    num_epochs = 100
    batch_size = 16
    lr = 0.0004
    val_split_ratio = 0.2
    spectral_loss_weight = 0.3

    # Model save
    model_dir = 'NB_SR_Model'
    results_dir = 'NB_SR_results'

    # Device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    num_visual_samples = 3

    base_features = 16
    transformer_dim = 96
    num_tokens = 64

config = Config()

class SpectralDataset(Dataset):
    def __init__(self, data_dir, target_bands=TARGET_BANDS, rgb_bands=RGB_BANDS,
                 augment=False, max_samples=None):
        self.data_dir = data_dir
        self.target_bands = target_bands
        self.rgb_bands = rgb_bands
        self.augment = augment
        self.max_samples = max_samples

        print(f"Scanning {data_dir}...")
        self.file_paths = []
        for ext in ['*.npy', '*.NPY']:
            self.file_paths.extend(glob.glob(os.path.join(data_dir, ext)))

        print(f"Found {len(self.file_paths)} files")
        if max_samples and len(self.file_paths) > max_samples:
            print(f"Limiting to {max_samples} samples for faster processing")
            self.file_paths = self.file_paths[:max_samples]

        self.data_cache = {}
        self.loaded_indices = []

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        if idx in self.data_cache:
            return self.data_cache[idx]

        file_path = self.file_paths[idx]

        try:
            data = np.load(file_path, allow_pickle=True).astype(np.float32)
            if data.shape != (64, 64, 153):
                if data.size == 64*64*153:
                    data = data.reshape(64, 64, 153)
                else:
                    raise ValueError(f"Incorrect size: {data.shape}")


            rgb = data[:, :, self.rgb_bands]
            hr = data[:, :, self.target_bands]

            rgb = rgb.transpose(2, 0, 1)
            hr = hr.transpose(2, 0, 1)

            rgb_tensor = torch.from_numpy(rgb.copy())
            hr_tensor = torch.from_numpy(hr.copy())


            def normalize_channel(channel):
                min_val = channel.min()
                max_val = channel.max()
                if max_val > min_val:
                    return (channel - min_val) / (max_val - min_val)
                return channel * 0

            for c in range(rgb_tensor.shape[0]):
                rgb_tensor[c] = normalize_channel(rgb_tensor[c])
            for c in range(hr_tensor.shape[0]):
                hr_tensor[c] = normalize_channel(hr_tensor[c])


            rgb_tensor = rgb_tensor + torch.randn_like(rgb_tensor) * 1e-6
            hr_tensor = hr_tensor + torch.randn_like(hr_tensor) * 1e-6

            if self.augment:
                rgb_tensor, hr_tensor = self.apply_augmentation(rgb_tensor, hr_tensor)

            self.data_cache[idx] = (rgb_tensor, hr_tensor)
            self.loaded_indices.append(idx)

            if len(self.data_cache) > 100:
                oldest_idx = self.loaded_indices.pop(0)
                del self.data_cache[oldest_idx]

            return rgb_tensor, hr_tensor

        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")
            return torch.zeros((NUM_INPUT_BANDS, 64, 64)), torch.zeros((NUM_TARGET_BANDS, 64, 64))

    def apply_augmentation(self, rgb, hr):
        if random.random() > 0.5:
            rgb = torch.flip(rgb, dims=[1])
            hr = torch.flip(hr, dims=[1])
        if random.random() > 0.5:
            rgb = torch.flip(rgb, dims=[2])
            hr = torch.flip(hr, dims=[2])
        return rgb, hr



class EnhancedResidualBlock(nn.Module):
    def __init__(self, channels, dilation=1, reduction=8):
        super(EnhancedResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3,
                              padding=dilation, dilation=dilation, bias=False)
        self.norm1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3,
                              padding=1, bias=False)
        self.norm2 = nn.BatchNorm2d(channels)
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention()
        self.activation = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.norm1(out)
        out = self.activation(out)
        out = self.conv2(out)
        out = self.norm2(out)
        out = self.channel_attention(out)
        out = self.spatial_attention(out)
        out = out + identity
        out = self.activation(out)
        return out

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        attention = self.sigmoid(avg_out + max_out)
        return x * attention

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size,
                             padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attention = self.conv(torch.cat([avg_out, max_out], dim=1))
        attention = self.sigmoid(attention)
        return x * attention

class MultiScaleBlock(nn.Module):
    def __init__(self, channels):
        super(MultiScaleBlock, self).__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(channels, channels//4, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels//4),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels//4, channels//4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels//4),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(channels, channels//4, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels//4),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels//4, channels//4, kernel_size=3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(channels//4),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(channels, channels//4, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels//4),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(channels//4, channels//4, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(channels//4),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.branch4 = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels//4, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels//4),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.1, inplace=True)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = F.interpolate(self.branch4(x), size=x.shape[2:], mode='bilinear', align_corners=False)
        out = torch.cat([b1, b2, b3, b4], dim=1)
        out = self.fusion(out)
        return out + x

class LightweightTransformer(nn.Module):
    def __init__(self, dim=96, num_tokens=64, num_heads=4, num_layers=2):
        super(LightweightTransformer, self).__init__()
        self.num_tokens = num_tokens
        self.dim = dim
        self.token_proj = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(dim),
            nn.LeakyReLU(0.1, inplace=True)
        )
        self.pos_embedding = nn.Parameter(torch.randn(1, num_tokens, dim) * 0.02)
        self.layers = nn.ModuleList([
            TransformerLayer(dim, num_heads) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        b, c, h, w = x.shape

        grid_size = int(math.sqrt(self.num_tokens))
        tokens = F.adaptive_avg_pool2d(x, (grid_size, grid_size))
        tokens = self.token_proj(tokens)
        tokens = tokens.flatten(2).transpose(1, 2)  # (B, num_tokens, C)
        tokens = tokens + self.pos_embedding
        for layer in self.layers:
            tokens = layer(tokens)
        tokens = self.norm(tokens)
        tokens = tokens.transpose(1, 2).reshape(b, self.dim, grid_size, grid_size)
        tokens = F.interpolate(tokens, size=(h, w), mode='bilinear', align_corners=False)
        return tokens

class TransformerLayer(nn.Module):
    def __init__(self, dim, num_heads):
        super(TransformerLayer, self).__init__()
        self.attention = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Linear(dim // 2, dim)
        )
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

class SpectralCorrelationLoss(nn.Module):
    def __init__(self, weight=0.3):
        super(SpectralCorrelationLoss, self).__init__()
        self.weight = weight

    def forward(self, pred, target):
        mse_loss = F.mse_loss(pred, target)
        b, c, h, w = pred.shape
        pred_flat = pred.view(b, c, -1)
        target_flat = target.view(b, c, -1)
        pred_corr = torch.bmm(pred_flat, pred_flat.transpose(1, 2)) / (h * w)
        target_corr = torch.bmm(target_flat, target_flat.transpose(1, 2)) / (h * w)
        corr_loss = F.mse_loss(pred_corr, target_corr)
        total_loss = mse_loss + self.weight * corr_loss
        return total_loss, mse_loss.item(), corr_loss.item()



class SophisticatedHybridUNet(nn.Module):
    """
    Hybrid U-Net as described in the text:
    - Input: 3-channel RGB (64x64)
    - Encoder: 3 downsampling stages (64→32→16→8) with channel doubling
    - Transformer bottleneck at 8x8 (64 tokens)
    - Decoder: 3 upsampling stages with skip connections
    - Output: 6 target spectral bands
    """
    def __init__(self, input_channels=3, output_channels=6, base_features=24):
        super(SophisticatedHybridUNet, self).__init__()
        self.base_features = base_features

        print(f"Sophisticated Hybrid UNet Configuration (Text-compliant):")
        print(f"  Input channels: {input_channels} (RGB)")
        print(f"  Output channels: {output_channels}")
        print(f"  Base features: {base_features}")
        print(f"  Encoder channels: {base_features} → {base_features*2} → {base_features*4} → {base_features*8}")


        self.initial_conv = nn.Sequential(
            nn.Conv2d(input_channels, base_features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_features),
            nn.LeakyReLU(0.1, inplace=True)
        )


        self.enc1_conv = nn.Sequential(
            EnhancedResidualBlock(base_features),
            MultiScaleBlock(base_features)
        )
        self.enc1_down = nn.Conv2d(base_features, base_features*2, kernel_size=3, stride=2, padding=1)


        self.enc2_conv = nn.Sequential(
            EnhancedResidualBlock(base_features*2),
            MultiScaleBlock(base_features*2)
        )
        self.enc2_down = nn.Conv2d(base_features*2, base_features*4, kernel_size=3, stride=2, padding=1)


        self.enc3_conv = nn.Sequential(
            EnhancedResidualBlock(base_features*4),
            MultiScaleBlock(base_features*4)
        )
        self.enc3_down = nn.Conv2d(base_features*4, base_features*8, kernel_size=3, stride=2, padding=1)


        self.bottleneck_conv = nn.Sequential(
            EnhancedResidualBlock(base_features*8),
            EnhancedResidualBlock(base_features*8, dilation=2)
        )


        self.bottleneck_proj = nn.Conv2d(base_features*8, config.transformer_dim, kernel_size=1, bias=False)
        self.transformer = LightweightTransformer(
            dim=config.transformer_dim,
            num_tokens=config.num_tokens,
            num_heads=4,
            num_layers=2
        )
        self.bottleneck_proj_back = nn.Conv2d(config.transformer_dim, base_features*8, kernel_size=1, bias=False)



        self.dec3_up = nn.ConvTranspose2d(base_features*8, base_features*4, kernel_size=2, stride=2)
        self.dec3_conv = nn.Sequential(
            nn.Conv2d(base_features*8, base_features*4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_features*4),
            nn.LeakyReLU(0.1, inplace=True),
            EnhancedResidualBlock(base_features*4),
            MultiScaleBlock(base_features*4)
        )


        self.dec2_up = nn.ConvTranspose2d(base_features*4, base_features*2, kernel_size=2, stride=2)
        self.dec2_conv = nn.Sequential(
            nn.Conv2d(base_features*4, base_features*2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_features*2),
            nn.LeakyReLU(0.1, inplace=True),
            EnhancedResidualBlock(base_features*2),
            MultiScaleBlock(base_features*2)
        )


        self.dec1_up = nn.ConvTranspose2d(base_features*2, base_features, kernel_size=2, stride=2)
        self.dec1_conv = nn.Sequential(
            nn.Conv2d(base_features*2, base_features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_features),
            nn.LeakyReLU(0.1, inplace=True),
            EnhancedResidualBlock(base_features),
            MultiScaleBlock(base_features)
        )

        # ==== FINAL OUTPUT ==========
        self.output_conv = nn.Sequential(
            nn.Conv2d(base_features, base_features, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(base_features),
            nn.LeakyReLU(0.1, inplace=True),
            ChannelAttention(base_features),
            nn.Conv2d(base_features, output_channels, kernel_size=1),
            nn.Sigmoid()
        )

        total_params = sum(p.numel() for p in self.parameters())
        print(f"  Total parameters: {total_params:,}")

    def forward(self, x):

        x0 = self.initial_conv(x)


        enc1 = self.enc1_conv(x0)
        pool1 = self.enc1_down(enc1)

        enc2 = self.enc2_conv(pool1)
        pool2 = self.enc2_down(enc2)

        enc3 = self.enc3_conv(pool2)
        pool3 = self.enc3_down(enc3)

        # Bottleneck
        bottleneck = self.bottleneck_conv(pool3)
        bottleneck_in = self.bottleneck_proj(bottleneck)
        transformer_out = self.transformer(bottleneck_in)
        bottleneck_out = self.bottleneck_proj_back(transformer_out)

        # Decoder
        up3 = self.dec3_up(bottleneck_out)
        up3 = torch.cat([up3, enc3], dim=1)
        dec3 = self.dec3_conv(up3)

        up2 = self.dec2_up(dec3)
        up2 = torch.cat([up2, enc2], dim=1)
        dec2 = self.dec2_conv(up2)

        up1 = self.dec1_up(dec2)
        up1 = torch.cat([up1, enc1], dim=1)
        dec1 = self.dec1_conv(up1)

        output = self.output_conv(dec1)
        return output


# TRAINING


def train_sophisticated_model():
    print(f"\nTraining Sophisticated Hybrid UNet on: {config.device}")
    print(f"Target bands: {TARGET_BANDS}")
    print(f"Spectral loss weight: {config.spectral_loss_weight}")

    Path(config.model_dir).mkdir(exist_ok=True)
    Path(config.results_dir).mkdir(exist_ok=True)

    dataset = SpectralDataset(config.train_data, augment=True, max_samples=2000)
    if len(dataset) == 0:
        print("No valid data found!")
        return None

    print(f"Loaded {len(dataset)} samples")
    val_size = int(len(dataset) * config.val_split_ratio)
    train_size = len(dataset) - val_size
    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"Training: {train_size} samples, Validation: {val_size} samples")

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    model = SophisticatedHybridUNet(
        input_channels=NUM_INPUT_BANDS,
        output_channels=NUM_TARGET_BANDS,
        base_features=config.base_features
    ).to(config.device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel Parameters:")
    print(f"  Total: {total_params:,} (<1.5M ✓)")
    print(f"  Trainable: {trainable_params:,}")
    print(f"  Memory: ~{total_params * 4 / 1024 / 1024:.2f} MB")

    spectral_loss_fn = SpectralCorrelationLoss(weight=config.spectral_loss_weight)
    mse_loss_fn = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=config.lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.num_epochs, eta_min=1e-6)

    patience = 12
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses, val_losses = [], []

    print("\n" + "="*50)
    print("Starting Sophisticated Hybrid UNet Training")
    print("="*50)

    for epoch in range(config.num_epochs):
        model.train()
        train_total_loss = 0.0
        train_mse_loss = 0.0
        train_spectral_loss = 0.0
        batch_count = 0

        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.num_epochs} [Train]')
        for lr_batch, hr_batch in train_bar:
            lr_batch = lr_batch.to(config.device)
            hr_batch = hr_batch.to(config.device)

            optimizer.zero_grad()
            outputs = model(lr_batch)
            total_loss, mse_loss, spectral_loss = spectral_loss_fn(outputs, hr_batch)
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()

            train_total_loss += total_loss.item()
            train_mse_loss += mse_loss
            train_spectral_loss += spectral_loss
            batch_count += 1

            train_bar.set_postfix({
                'total': total_loss.item(),
                'mse': mse_loss,
                'spectral': spectral_loss
            })

        avg_train_loss = train_total_loss / batch_count
        avg_train_mse = train_mse_loss / batch_count
        avg_train_spectral = train_spectral_loss / batch_count
        train_losses.append(avg_train_loss)

        model.eval()
        val_total_loss = 0.0
        val_batch_count = 0
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{config.num_epochs} [Val]')
            for lr_batch, hr_batch in val_bar:
                lr_batch = lr_batch.to(config.device)
                hr_batch = hr_batch.to(config.device)
                outputs = model(lr_batch)
                loss = mse_loss_fn(outputs, hr_batch)
                val_total_loss += loss.item()
                val_batch_count += 1
                val_bar.set_postfix({'loss': loss.item()})

        avg_val_loss = val_total_loss / val_batch_count
        val_losses.append(avg_val_loss)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        print(f"\nEpoch {epoch+1}:")
        print(f"  Train - Total: {avg_train_loss:.6f}, MSE: {avg_train_mse:.6f}, Spectral: {avg_train_spectral:.6f}")
        print(f"  Val MSE: {avg_val_loss:.6f}, LR: {current_lr:.6f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_val_loss,
                'train_losses': train_losses,
                'val_losses': val_losses,
            }, os.path.join(config.model_dir, 'best_model.pth'))
            print(f"  ✓ Saved best model (loss: {best_val_loss:.6f})")
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(),
                      os.path.join(config.model_dir, f'checkpoint_epoch_{epoch+1}.pth'))

        if patience_counter >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

    torch.save(model.state_dict(), os.path.join(config.model_dir, 'final_model.pth'))
    print(f"\n✓ Training completed! Best val loss: {best_val_loss:.6f}")

    plot_training_history(train_losses, val_losses)
    return model

def plot_training_history(train_losses, val_losses):
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, 'b-', label='Training Loss', linewidth=2)
    plt.plot(val_losses, 'r-', label='Validation Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Sophisticated Hybrid UNet Training History')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig(os.path.join(config.results_dir, 'training_history.png'), dpi=150)
    plt.close()


# ---------EVALUATION

def evaluate_water_bodies(model):
    print(f"\n" + "="*60)
    print(f"Evaluating Sophisticated Hybrid UNet on Water Bodies")
    print("="*60)

    if not os.path.exists(config.water_bodies_dir):
        print(f"Water bodies directory not found: {config.water_bodies_dir}")
        return None, None

    dataset = SpectralDataset(config.water_bodies_dir, augment=False, max_samples=50)
    if len(dataset) == 0:
        print("No water bodies data found!")
        return None, None

    print(f"Loaded {len(dataset)} water bodies samples")
    dataloader = DataLoader(
        dataset,
        batch_size=min(config.batch_size, len(dataset)),
        shuffle=False,
        num_workers=0
    )

    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for lr_batch, hr_batch in tqdm(dataloader, desc="Evaluating"):
            lr_batch = lr_batch.to(config.device)
            hr_batch = hr_batch.to(config.device)
            predictions = model(lr_batch)
            all_predictions.append(predictions.cpu())
            all_targets.append(hr_batch.cpu())

    if not all_predictions:
        print("No predictions made!")
        return None, None

    predictions = torch.cat(all_predictions, dim=0)
    targets = torch.cat(all_targets, dim=0)

    mse = torch.mean((predictions - targets) ** 2).item()
    mae = torch.mean(torch.abs(predictions - targets)).item()
    rmse = torch.sqrt(torch.mean((predictions - targets) ** 2)).item()
    psnr = 10 * torch.log10(torch.tensor(1.0) / mse).item() if mse > 0 else float('inf')

    print(f"\nEvaluation Metrics:")
    print(f"MSE:   {mse:.6f}")
    print(f"MAE:   {mae:.6f}")
    print(f"RMSE:  {rmse:.6f}")
    print(f"PSNR:  {psnr:.2f} dB")

    metrics_file = os.path.join(config.results_dir, 'water_bodies_metrics.txt')
    with open(metrics_file, 'w') as f:
        f.write(f"Water Bodies Evaluation - Sophisticated Hybrid UNet\n")
        f.write("="*50 + "\n")
        f.write(f"Samples evaluated: {len(predictions)}\n\n")
        f.write(f"MSE:  {mse:.6f}\n")
        f.write(f"MAE:  {mae:.6f}\n")
        f.write(f"RMSE: {rmse:.6f}\n")
        f.write(f"PSNR: {psnr:.2f} dB\n")

    create_visualization(predictions, targets, mse, mae, rmse, psnr)
    return predictions, targets

def create_visualization(predictions, targets, mse, mae, rmse, psnr):
    print("\nCreating visualizations...")
    vis_dir = os.path.join(config.results_dir, 'water_bodies_visualizations')
    Path(vis_dir).mkdir(exist_ok=True)

    fig, axes = plt.subplots(2, NUM_TARGET_BANDS, figsize=(18, 6))
    for band_idx in range(NUM_TARGET_BANDS):
        target_img = targets[0, band_idx].numpy()
        axes[0, band_idx].imshow(target_img, cmap='viridis', vmin=0, vmax=1)
        axes[0, band_idx].set_title(f'Target B{TARGET_BANDS[band_idx]}')
        axes[0, band_idx].axis('off')

        pred_img = predictions[0, band_idx].numpy()
        axes[1, band_idx].imshow(pred_img, cmap='viridis', vmin=0, vmax=1)
        axes[1, band_idx].set_title(f'Pred B{TARGET_BANDS[band_idx]}')
        axes[1, band_idx].axis('off')

    metrics_text = f'MSE: {mse:.4f}, MAE: {mae:.4f}, PSNR: {psnr:.1f} dB'
    plt.suptitle(f'Sophisticated Hybrid UNet - Water Body Sample 1\n{metrics_text}', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, 'sample_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()

    avg_target = targets.mean(dim=(0, 2, 3))
    avg_pred = predictions.mean(dim=(0, 2, 3))
    plt.figure(figsize=(10, 6))
    band_indices = range(NUM_TARGET_BANDS)
    plt.plot(band_indices, avg_target.numpy(), 'b-o', linewidth=3, markersize=8, label='Target')
    plt.plot(band_indices, avg_pred.numpy(), 'r--s', linewidth=3, markersize=8, label='Predicted')
    plt.fill_between(band_indices, avg_target.numpy(), avg_pred.numpy(), alpha=0.2, color='gray')
    plt.xlabel('Band Index')
    plt.ylabel('Normalized Intensity')
    plt.title('Average Spectral Profile - Sophisticated Hybrid UNet', fontsize=14)
    plt.xticks(band_indices, [f'B{b}' for b in TARGET_BANDS])
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, 'spectral_profile.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Visualizations saved to: {vis_dir}")


# ----------------MAIN

def main():
    print("="*70)
    print("SOPHISTICATED HYBRID UNET - RGB to Hyperspectral Reconstruction")
    print("="*70)
    print(f"Input: RGB (bands {RGB_BANDS}) → Output: {NUM_TARGET_BANDS} bands {TARGET_BANDS}")
    print(f"Model Components:")
    print(f"  • 3 Encoder / 3 Decoder stages")
    print(f"  • Advanced Residual Blocks")
    print(f"  • Channel & Spatial Attention")
    print(f"  • Multi-scale Processing")
    print(f"  • Lightweight Transformer (64 tokens at 8×8)")
    print(f"  • Infused Spectral Loss")
    print(f"Training: {config.num_epochs} epochs, batch size {config.batch_size}")
    print(f"Spectral loss weight: {config.spectral_loss_weight}")
    print("="*70)

    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)

    if config.device == 'cuda':
        print(f"Using GPU: {torch.cuda.get_device_name(0)}")

    print("\n[1/2] Training Sophisticated Hybrid UNet Model...")
    model = train_sophisticated_model()
    if model is None:
        print("Training failed!")
        return

    print("\n[2/2] Loading Best Model for Evaluation...")
    best_model_path = os.path.join(config.model_dir, 'best_model.pth')
    if os.path.exists(best_model_path):
        print(f"Loading best model from: {best_model_path}")
        checkpoint = torch.load(best_model_path, map_location=config.device)
        model = SophisticatedHybridUNet(
            input_channels=NUM_INPUT_BANDS,
            output_channels=NUM_TARGET_BANDS,
            base_features=config.base_features
        ).to(config.device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Best model epoch: {checkpoint['epoch']+1}, loss: {checkpoint['loss']:.6f}")
    else:
        print("Best model not found, using current model")

    evaluate_water_bodies(model)

    print("\n" + "="*70)
    print("SOPHISTICATED HYBRID UNET TRAINING AND EVALUATION COMPLETED!")
    print("="*70)
    print(f"Results saved in: {config.results_dir}")
    print(f"Models saved in: {config.model_dir}")
    print("="*70)

if __name__ == "__main__":
    main()

SOPHISTICATED HYBRID UNET - RGB to Hyperspectral Reconstruction
Input: RGB (bands [10, 20, 30]) → Output: 6 bands [21, 24, 27, 28, 29, 30]
Model Components:
  • 3 Encoder / 3 Decoder stages
  • Advanced Residual Blocks
  • Channel & Spatial Attention
  • Multi-scale Processing
  • Lightweight Transformer (64 tokens at 8×8)
  • Infused Spectral Loss
Training: 100 epochs, batch size 16
Spectral loss weight: 0.3

[1/2] Training Sophisticated Hybrid UNet Model...

Training Sophisticated Hybrid UNet on: cpu
Target bands: [21, 24, 27, 28, 29, 30]
Spectral loss weight: 0.3
Scanning /content/drive/MyDrive/Colab Notebooks/Hyperion/train_images...
Found 2464 files
Limiting to 2000 samples for faster processing
Loaded 2000 samples
Training: 1600 samples, Validation: 400 samples
Sophisticated Hybrid UNet Configuration (Text-compliant):
  Input channels: 3 (RGB)
  Output channels: 6
  Base features: 16
  Encoder channels: 16 → 32 → 64 → 128
  Total parameters: 1,222,950

Model Parameters:
  Total: 

Epoch 1/100 [Train]:   0%|          | 0/100 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Epoch 1/100 [Train]:   0%|          | 0/100 [00:08<?, ?it/s]


KeyboardInterrupt: 